In [ ]:
# Open shapefile
# Get rid of overlapping ones

# For each shapefile find the images which it intersects 
# Merge together 

# Find the centroid of the shapefile
# buffer out the shapefile by X metres, make a rectangle, then create a square of the maximum pixel size we will show

# Clip the imagery to this

# Create the images required 

# Create the link to the google maps version

# Create JSON with links to the files and save out


In [19]:
from pathlib import Path
import json
import math
import shutil

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.io import MemoryFile
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.features import rasterize
from shapely.geometry import box
from PIL import Image, ImageDraw
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import matplotlib.colors as colors

import os

from pandas import cut

In [20]:
def crop_and_resize_review_image(path, out_path=None):
    img = Image.open(path).convert("RGB")

    width, height = img.size

    cropped = img.crop((6, 6, width - 6, height - 6))

    resized = cropped.resize((width, height), Image.Resampling.NEAREST)

    out_path = out_path or path
    resized.save(out_path)

def remove_overlapping_polygons(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Removes polygons that overlap any earlier polygon.
    Keeps the first occurrence.
    """
    keep_rows = []
    accepted_geoms = []

    for idx, row in gdf.iterrows():
        geom = row.geometry

        overlaps_existing = any(
            geom.intersects(existing) and geom.intersection(existing).area > 0
            for existing in accepted_geoms
        )

        if not overlaps_existing:
            keep_rows.append(idx)
            accepted_geoms.append(geom)

    return gdf.loc[keep_rows].copy().reset_index(drop=True)

def create_spatial_index_from_directory(tile_directory = None, show_progress = True, valid_formats=None):
   
    # Check if file formats in directory are ones that can be processed
    formats_in_directory = set([x.suffix for x in tile_directory.glob("*")])
    __valid_formats_string__ = (", ".join(sorted(valid_formats)))
    
    # Any invalid formats found 
    invalid_format_count = ([test_format in valid_formats for test_format in formats_in_directory]).count(False)
    valid_format_count = len(formats_in_directory) - invalid_format_count
    
    # If no valid formats then raise error and quit
    if valid_format_count == 0: 
        print(f"No supported file types found to build spatial index, spatial indexs can only currently be created for {__valid_formats_string__}")
    # If invalid formats found then report error and continue
    if invalid_format_count != 0:
        print((f"Unsupported file types found in directory, spatial indexs can only currently be created for {__valid_formats_string__}, other file formats will be ignored"))

    # Create spatial index 
    # Get all files for accepted formats 
    files = list(tile_directory.glob("*.tif"))
    iterator = tqdm(files, desc="Building Spatial Index") if show_progress else files
    
    # Loop over each item and add to spatial index file 
    spatial_index = []
    for item in iterator:
        with rasterio.open(item) as src:
            spatial_index.append({
                "path": item,
                "crs": src.crs.to_string() if src.crs else None, 
                "geometry": box(*src.bounds)
            })
    # Convert to geodataframe and return 
    return gpd.GeoDataFrame(spatial_index, geometry="geometry", crs=spatial_index[0]["crs"])

def make_adaptive_review_square(
    geom,
    context_buffer_m=500,
    min_side_m=1000,
    max_side_m=None
):
    """
    Creates a square that always contains the full site plus context.

    context_buffer_m = extra space around the polygon
    min_side_m       = minimum square size for small sites
    max_side_m       = optional cap to avoid huge review images
    """
    buffered = geom.buffer(context_buffer_m)

    minx, miny, maxx, maxy = buffered.bounds

    width = maxx - minx
    height = maxy - miny

    side = max(width, height, min_side_m)

    if max_side_m is not None:
        side = min(side, max_side_m)

    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2
    half = side / 2

    return box(
        cx - half,
        cy - half,
        cx + half,
        cy + half
    )

def find_intersecting_tiles(tile_index: gpd.GeoDataFrame, geom):
    matches = tile_index[tile_index.intersects(geom)]
    return [Path(p) for p in matches["path"].tolist()]

def merge_tiles(tile_paths):
    """
    Merge all intersecting tiles into an in-memory raster.
    """
    srcs = [rasterio.open(p) for p in tile_paths]

    try:
        mosaic, transform = merge(srcs)
        profile = srcs[0].profile.copy()
        profile.update({
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": transform,
            "count": mosaic.shape[0],
        })

        return mosaic, profile

    finally:
        for src in srcs:
            src.close()

def clip_mosaic_to_square_800(mosaic, profile, square_geom, final_size_px=800):
    """
    Clips mosaic to square geometry, then resamples to final_size_px x final_size_px.
    Returns array shaped (bands, 800, 800).
    """
    with MemoryFile() as memfile:
        with memfile.open(**profile) as dataset:
            dataset.write(mosaic)

            clipped, clipped_transform = mask(
                dataset,
                [square_geom],
                crop=True,
                filled=True,
                nodata=profile.get("nodata", 0),
            )

            clipped_profile = dataset.profile.copy()
            clipped_profile.update({
                "height": clipped.shape[1],
                "width": clipped.shape[2],
                "transform": clipped_transform,
            })

    bands, h, w = clipped.shape

    resampled = np.empty(
        shape=(bands, final_size_px, final_size_px),
        dtype=np.float32
    )

    with MemoryFile() as memfile:
        temp_profile = clipped_profile.copy()
        temp_profile.update({
            "height": h,
            "width": w,
            "count": bands,
            "dtype": clipped.dtype,
        })

        with memfile.open(**temp_profile) as src:
            src.write(clipped)

            for b in range(1, bands + 1):
                resampled[b - 1] = src.read(
                    b,
                    out_shape=(final_size_px, final_size_px),
                    resampling=Resampling.nearest,
                )

    return resampled

def normalize_to_uint8(arr, nodata=None, pmin=2, pmax=98):
    """
    Percentile stretch single-band array to uint8.
    """
    arr = arr.astype(np.float32)

    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)

    valid = np.isfinite(arr)

    if valid.sum() == 0:
        return np.zeros(arr.shape, dtype=np.uint8)

    lo, hi = np.nanpercentile(arr[valid], [pmin, pmax])

    if hi <= lo:
        return np.zeros(arr.shape, dtype=np.uint8)

    stretched = (arr - lo) / (hi - lo)
    stretched = np.clip(stretched, 0, 1)

    stretched[~valid] = 0

    return (stretched * 255).astype(np.uint8)

def save_band_images(site_id, clipped_arr, out_dir: Path, band_names=None):
    """
    Save each band as a separate PNG.
    """
    out_paths = []

    n_bands = clipped_arr.shape[0]

    if band_names is None:
        band_names = [f"band_{i+1}" for i in range(n_bands)]

    for i, band_name in enumerate(band_names):
        img_arr = normalize_to_uint8(clipped_arr[i])

        filename = f"{site_id}_{band_name}.png"
        out_path = out_dir / filename

        Image.fromarray(img_arr).save(out_path)

        out_paths.append({
            "band": band_name,
            "path": f"images/{filename}"
        })

    return out_paths

def google_maps_link(square_geom, source_crs):
    """
    Create Google Maps link centred on review square.
    """
    square_wgs84 = (
        gpd.GeoSeries([square_geom], crs=source_crs)
        .to_crs("EPSG:4326")
        .iloc[0]
    )

    centroid = square_wgs84.centroid
    lon, lat = centroid.x, centroid.y
    
    #f"https://www.google.com/maps/@{lat},{lon},16z/data=!3m1!1e3"
    return f"https://maps.google.com/?q={lat}, {lon}"

def stretch_to_01(arr, vmin=None, vmax=None, std_multiplier=1.0):
    arr = arr.astype("float32")
    valid = np.isfinite(arr)

    if valid.sum() == 0:
        return np.zeros_like(arr, dtype="float32")

    mean = np.nanmean(arr[valid])
    std = np.nanstd(arr[valid])

    if vmin is None:
        vmin = mean - std_multiplier * std
    if vmax is None:
        vmax = mean + std_multiplier * std

    if vmax <= vmin:
        return np.zeros_like(arr, dtype="float32")

    out = (arr - vmin) / (vmax - vmin)
    return np.clip(out, 0, 1)

def get_default_vmin_vmax(arr, vmin=None, vmax=None, std_multiplier=1.0):
    arr = arr.astype("float32")
    valid = np.isfinite(arr)

    if valid.sum() == 0:
        return 0.0, 1.0

    mean = np.nanmean(arr[valid])
    std = np.nanstd(arr[valid])

    if vmin is None:
        vmin = mean - std_multiplier * std
    if vmax is None:
        vmax = mean + std_multiplier * std

    return float(vmin), float(vmax)

def save_rgb_image(arr, band_indices, out_path, fixed_ranges=None):
    """
    arr shape: (bands, height, width)
    band_indices: e.g. [3, 2, 1] using zero-based indices
    fixed_ranges: optional [(r_min, r_max), (g_min, g_max), (b_min, b_max)]
    """
    rgb = []

    for i, band_idx in enumerate(band_indices):
        band = arr[band_idx]

        if fixed_ranges is not None:
            vmin, vmax = fixed_ranges[i]
            band_01 = stretch_to_01(band, vmin=vmin, vmax=vmax)
        else:
            band_01 = stretch_to_01(band)

        rgb.append(band_01)

    rgb = np.stack(rgb, axis=-1)
    rgb = np.nan_to_num(rgb, nan=0.0, posinf=1.0, neginf=0.0)

    rgb_uint8 = np.clip(rgb * 255, 0, 255).astype("uint8")

    Image.fromarray(rgb_uint8).save(out_path)

def save_colormap_image(
    band_arr,
    out_path,
    cmap="viridis",
    vmin=None,
    vmax=None,
    label="Value",
    save_colorbar=True, 
    std_multiplier=1.0, 
):
    vmin, vmax = get_default_vmin_vmax(
        band_arr,
        vmin=vmin,
        vmax=vmax,
        std_multiplier=std_multiplier
    )
    
    norm = colors.Normalize(vmin=vmin, vmax=vmax)
    cmap_obj = plt.get_cmap(cmap)

    rgba = cmap_obj(norm(band_arr))
    rgb = (rgba[:, :, :3] * 255).astype("uint8")
    Image.fromarray(rgb).save(out_path)

    colorbar_path = None

    if save_colorbar:
        colorbar_path = out_path.with_name(out_path.stem + "_scale.png")

        fig, ax = plt.subplots(figsize=(6, 0.6))
        fig.subplots_adjust(bottom=0.5)

        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap_obj)
        sm.set_array([])

        cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")
        cbar.set_label(label)

        fig.savefig(colorbar_path, dpi=150, bbox_inches="tight", pad_inches=0)
        plt.close(fig)

    return {
        "image": out_path,
        "colorbar": colorbar_path,
        "vmin": float(vmin),
        "vmax": float(vmax),
        "cmap": cmap
    }

def save_greyscale_image(
    band_arr,
    out_path,
    vmin=None,
    vmax=None,
    pmin=2,
    pmax=98,
    label="Value",
    save_colorbar=True
):
    result = save_colormap_image(
        band_arr=band_arr,
        out_path=out_path,
        cmap="gray",
        vmin=vmin,
        vmax=vmax,
        pmin=pmin,
        pmax=pmax,
        label=label,
        save_colorbar=save_colorbar
    )

    return result

def polygon_border_mask(geom, square_geom, size_px=800, border_width_px=3):
    minx, miny, maxx, maxy = square_geom.bounds

    transform = rasterio.transform.from_bounds(
        minx, miny, maxx, maxy,
        size_px, size_px
    )

    polygon_mask = rasterize(
        [(geom, 1)],
        out_shape=(size_px, size_px),
        transform=transform,
        fill=0,
        dtype="uint8",
        all_touched=True
    )

    from scipy.ndimage import binary_dilation

    edge = polygon_mask.astype(bool) ^ binary_dilation(
        polygon_mask.astype(bool),
        iterations=border_width_px
    )

    return edge

def add_polygon_border_to_png(
    image_path,
    geom,
    square_geom,
    output_path=None,
    size_px=800,
    border_width_px=3,
    border_rgb=(255, 255, 255)
):
    if output_path is None:
        output_path = image_path

    img = Image.open(image_path).convert("RGB")
    arr = np.array(img)

    edge = polygon_border_mask(
        geom=geom,
        square_geom=square_geom,
        size_px=size_px,
        border_width_px=border_width_px
    )

    arr[edge] = border_rgb

    Image.fromarray(arr).save(output_path)

def render_site_images(site_id, clipped_800, recipes, out_dir, site_geom, square_geom):
    outputs = []

    for recipe in recipes:
        name = recipe["name"]
        img_path = out_dir / f"{site_id}_{name}.png"

        if recipe["type"] == "rgb":
            save_rgb_image(
                arr=clipped_800,
                band_indices=recipe["bands"],
                out_path=img_path,
                fixed_ranges=recipe.get("fixed_ranges")
            )

            outputs.append({
                "name": name,
                "type": "rgb",
                "path": f"imgs/review_images/{img_path.name}",
                "bands": recipe["bands"]
            })

        elif recipe["type"] == "colormap":
            result = save_colormap_image(
                band_arr=clipped_800[recipe["band"]],
                out_path=img_path,
                cmap=recipe.get("cmap", "viridis"),
                vmin=recipe.get("vmin"),
                vmax=recipe.get("vmax"),
                label=recipe.get("label", name),
                save_colorbar=True
            )

            outputs.append({
                "name": name,
                "type": "colormap",
                "path": f"imgs/review_images/{img_path.name}",
                "scale_path": f"imgs/review_images/{result['colorbar'].name}",
                "band": recipe["band"],
                "vmin": result["vmin"],
                "vmax": result["vmax"],
                "cmap": result["cmap"]
            })

        elif recipe["type"] == "greyscale":
            result = save_greyscale_image(
                band_arr=clipped_800[recipe["band"]],
                out_path=img_path,
                vmin=recipe.get("vmin"),
                vmax=recipe.get("vmax"),
                label=recipe.get("label", name),
                save_colorbar=True
            )

            outputs.append({
                "name": name,
                "type": "greyscale",
                "path": f"imgs/review_images/{img_path.name}",
                "scale_path": f"imgs/review_images/{result['colorbar'].name}",
                "band": recipe["band"],
                "vmin": result["vmin"],
                "vmax": result["vmax"]
            })

        if recipe.get("show_polygon", False):
            add_polygon_border_to_png(
                image_path=img_path,
                geom=site_geom,
                square_geom=square_geom,
                output_path=img_path,
                size_px=800,
                border_width_px=recipe.get("border_width_px", 4),
                border_rgb=recipe.get("border_rgb", (0, 0, 0))
            )   
        # finally crop image down to get rid of black edge
        crop_and_resize_review_image(img_path)    

    return outputs

In [24]:
# Config
MASTER_SHAPEFILE = Path(r"E:\bsc\Archaeological_Mound_Detection_India\data\deep_learning_outputs\post_processing\master_shapefile_20260518.shp")
TILES_DIR = Path(r"E:\bsc\Archaeological_Mound_Detection_India\data\new_tiles\ard")
OUT_DIR = Path(r"D:\bsc\Archaeological_Mound_Detection_India\6_expert_review_tool")

APP_IMAGE_DIR = OUT_DIR / "imgs/review_images"
APP_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

TILE_GLOB = "*.tif"

TARGET_CRS = "EPSG:32642"   # or whatever projected CRS your imagery uses
BUFFER_M = 500              # buffer around centroid/site
FINAL_SIZE_PX = 800         # output image size
SITE_ID_FIELD = "site_id"   # change to your ID field
PROB_FIELD = "median_pro"
PROB_STD_FIELD = "std_dev_pr"
 

APP_VERSION = "0.1.0"
BATCH_ID = "batch_001"

IMAGE_RECIPES = [
    {
        "name": "rgb",
        "type": "rgb",
        "bands": [2, 1, 0],
        "show_polygon": True,
        "fixed_ranges": [(0, 0.3), (0, 0.3), (0, 0.3)]
    },
    {
        "name": "SAVI (Wet)",
        "type": "colormap",
        "band": 23,
        "cmap": "RdYlGn",
        "label": "SAVI (Red = Not much vegetation, Green = Lots of vegetation)",
        "show_polygon": True,
        "std_multiplier": 3.0,
        "vmin": -0.5,
        "vmax": 0.5,
    },
    {
        "name": "MSRM",
        "type": "colormap",
        "band": 42,
        "cmap": "RdYlGn",
        "label": "MSRM (Red = Local low, Green = Local high)",
        "show_polygon": True,
        "std_multiplier": 2.0,
        "vmin": None,
        "vmax": None
    },
    {
        "name": "Clay Mineral Ratio",
        "type": "colormap",
        "band": 25,
        "cmap": "RdYlGn",
        "label": "Clay Mineral Ratio (Red = Low Clay, Green = High Clay)",
        "show_polygon": True,
        "std_multiplier": 3.0,
        "vmin": None,
        "vmax": None,
    },
]

In [29]:
# Open all sites and ensure that they are in the correct projection, and ensure that sites field exitss
sites = gpd.read_file(MASTER_SHAPEFILE)

if sites.crs is None:
    raise ValueError("Master shapefile has no CRS defined.")

sites = sites.to_crs(TARGET_CRS)
sites = sites.reset_index(drop=True)

if SITE_ID_FIELD not in sites.columns:
    sites[SITE_ID_FIELD] = [f"site_{i:04d}" for i in range(len(sites))]

sites_clean = remove_overlapping_polygons(sites)

print(f"Original sites: {len(sites)}")
print(f"After removing overlaps: {len(sites_clean)}")

# Create spatial index for fast lookup of tiles
tile_index = create_spatial_index_from_directory(TILES_DIR, TILE_GLOB, [".tif"])
tile_index.head()

review_records = []

sites_temp = sites_clean[1:21]

for idx, row in sites_temp.iterrows():
    # Get site id from row
    site_id = str(row[SITE_ID_FIELD])

    # Get median probability, probability std_dev
    median_prob = str(row[PROB_FIELD]*100)
    median_prob_std = str(row[PROB_STD_FIELD])

    # convert the median probability into a category 
    labels = ["Very Low", "Low", "Medium", "High", "Very High"]
    median_prob_category = cut([float(median_prob)], range(0, 105, 20), right=False, labels=labels)
    median_prob_category_label = median_prob_category[0]

    # Get row geometry 
    geom = row.geometry

    # Report, change to TQDM [TODO]
    print(f"Processing {site_id} ({idx}/{len(sites_temp)})")

    # Using the site geometry, make a square around the site 
    square_geom = make_adaptive_review_square(geom, context_buffer_m=500, min_side_m=1000, max_side_m=4000)
    side_m = square_geom.bounds[2] - square_geom.bounds[0]
    display_pixel_size_m = side_m / 800

    # Find all tiles that intersect with this square geom 
    intersecting_tiles = find_intersecting_tiles(tile_index, square_geom)

    # Sense check to ensure at least one tile is found 
    if len(intersecting_tiles) == 0:
        print(f"  No intersecting tiles found for {site_id}")
        continue
    
    # Mosaic images together 
    mosaic, profile = merge_tiles(intersecting_tiles)

    # Clip the mosaic to exactly 800 pixels square to ensure consistent images 
    clipped_800 = clip_mosaic_to_square_800(
        mosaic=mosaic,
        profile=profile,
        square_geom=square_geom,
        final_size_px=FINAL_SIZE_PX
    )
    

    image_outputs = render_site_images(
        site_id=site_id,
        clipped_800=clipped_800,
        recipes=IMAGE_RECIPES,
        out_dir=APP_IMAGE_DIR,
        site_geom=geom,
        square_geom=square_geom
    )

    # Create url based on centroid  
    maps_url = google_maps_link(square_geom, TARGET_CRS)

    # Create json record 
    record = {
        "candidate_id": site_id,
        "median_probability": median_prob,
        "median_probability_category": median_prob_category_label,
        "image_size_px": 800,
        "review_square_bounds": list(square_geom.bounds),
        "review_square_side_m": side_m,
        "display_pixel_size_m": display_pixel_size_m,
        "images": image_outputs,
        "google_maps_url": maps_url,
    }

    review_records.append(record)

json_path = OUT_DIR / "candidates.json"
js_path = OUT_DIR / "candidates.js"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(review_records, f, indent=2)

with json_path.open("r", encoding="utf-8") as f:
    candidates = json.load(f)

js_text = (
    "// Auto-generated from candidates.json\n"
    "// Do not edit this file directly; edit candidates.json and regenerate.\n\n"
    "const candidates = "
    + json.dumps(candidates, indent=2, ensure_ascii=False)
    + ";\n"
)

js_path.write_text(js_text, encoding="utf-8")

print(f"Created {js_path}")
print(f"Loaded {len(candidates)} candidates")

os.remove(json_path)


Original sites: 162
After removing overlaps: 159
Unsupported file types found in directory, spatial indexs can only currently be created for .tif, other file formats will be ignored


Building Spatial Index:   0%|          | 0/289 [00:00<?, ?it/s]

Processing 4e6a7727-ba03-46c4-86f7-63e2969c0e7f (1/20)
Processing f54a3652-6ad0-4597-9903-3dd5a29484bd (2/20)
Processing 62d9814d-f0d7-4013-82dc-fd8465a7331c (3/20)
Processing b9ab1967-7e64-4b6a-822c-099e04846fc2 (4/20)
Processing 059bffb5-bf8b-4051-8ac8-35e7c3bf8dc0 (5/20)
Processing 26ce7b1c-c6bc-4c45-b916-908e747b7401 (6/20)
Processing 0843bc84-b1a8-43ca-ad50-fcb40ad047f1 (7/20)
Processing ef439ec3-a802-4687-a2b4-25de4e871753 (8/20)
Processing ae382bb0-6e4d-46a0-a43b-0589b2d301a5 (9/20)
Processing d9b65393-2998-4b9b-b18b-abc9af9af63f (10/20)
Processing 9959c68a-78c6-4325-b8ae-d8ad151c7571 (11/20)
Processing 07eb1861-9689-4ca8-a2fb-b36e5677b322 (12/20)
Processing 90048482-726b-4c12-ac84-d537c2b5eba2 (13/20)
Processing 08396dc3-9bd1-44de-826e-74e47d3f8524 (14/20)
Processing c1ab752f-9cb1-4f04-9687-1cb71e30603d (15/20)
Processing 1eccff6e-4606-4b7a-9aae-5bca446e7eea (16/20)
Processing ac439c99-99b2-45f8-9910-4e35c4ffc03e (17/20)
Processing 7919817f-cf48-4b71-a19b-b3c03edd6239 (18/20)
P